# End-to-end candidate generation on 1N8Z

From the FoldX alanine scan to a ranked set of multi-substitution candidates, using
`design_pipeline` as it stands. No package code is changed here; where the API does not
expose something this notebook needs, the markdown says so instead of reaching in.

**This notebook needs a GPU box and the model environments.** There is no fixture mode.
The preflight cell probes exactly the adapters named in the parameters and stops if the
one `SCORER` names does not resolve, so a missing environment costs ten seconds rather
than an hour.

**Every intermediate table is written to disk, and every part reloads from disk.** No part
reads a variable another part defined: the handles are recorded in `pointers.json` under
the run directory and rebuilt from the parquet. So any part can be re-run on its own while
the filters are being tuned - edit the parameters cell, re-run setup, re-run that part.

| Part | What it does | Writes |
|---|---|---|
| 0 | parameters, preflight, parent complex | `pointers.json` |
| 1 | position selection from the alanine scan: distance, three tiers | `01_scan_with_distance.csv`, `01_tiers.csv` |
| 2 | `enumerate_variants` over those positions, per-position alphabets | `02_singles.parquet` |
| 3 | `score_variants`: `SCORER` plus runners-up, one column each | `03_scored_singles.parquet` |
| 4 | `gate`: liabilities, failures dropped | `04_survivors.parquet`, `04_annotated.parquet` |
| 5 | `propose_sets`, additive first pass, re-score the top sets | `05_sets.parquet`, `05_rescored.parquet` |
| 6 | the ranked output and a manifest of every row count | `06_candidates.csv`, `manifest.json` |

Row counts are printed at every stage with a `[rows]` prefix, so a filter that quietly
emptied the table is visible in the output rather than three parts later.

In [ ]:
# --- parameters ---------------------------------------------------------------
# The only cell meant to be edited. Paths are relative to the repository root.

RUN = "runs/candidate_generation"

# The parent. 1N8Z is a Fab - VH+CH1 as chain B, VL+CL as chain A - against the
# HER2 ectodomain as chain C. The established mapping for this campaign is
# H -> B, L -> A, antigen -> C; `chains=` below is checked against the file.
STRUCTURE_SOURCE = "data/structures/1N8Z.pdb"
HEAVY, LIGHT, ANTIGEN = "B", "A", "C"

# The alanine scan. 227 positions of the two variable domains, 3 FoldX runs each,
# with the 10-run re-runs from docs/track_a_filtering.md in `ddg_bind_doc`.
SCAN_CSV = "trastuzumab_foldx_alanine_scan.csv"

# Which ddG column the tiers are cut on. `ddg_bind_doc` is a 10-run mean where it
# exists and `ddg_bind` a 3-run one, so "best available" prefers the doc value and
# falls back to the raw column. Part 1 prints which positions move between the two;
# set this to "ddg_bind" to tier on the raw 3-run column alone.
TIER_ON = "ddg_bind_best"        # or "ddg_bind"

# Part 1, the tiers.
TIER1_MAX_DISTANCE = 5.0         # angstroms, minimum heavy atom to any ANTIGEN atom
TIER1_DDG = (0.2, 1.0)           # inclusive band: at the interface, contributing, not load-bearing
TIER2_DDG = 1.0                  # at or above this is a hotspot: conservative substitutions only
# Literature positions, carried regardless of what the scan says about them.
TIER3 = ("B:D31", "B:D102", "A:D28", "B:G101", "B:M107")
# A live negative control. F and W at VL S50 are measured losses, so if either
# scores well mid-run that is a fact about the scorer, not about the design.
CONTROLS = ("A:S50F", "A:S50W")

# Tier 2 substitutions. A hotspot is a position whose side chain carries binding
# energy, so it is restricted to substitutions that keep the feature the scan says
# is load-bearing: Y -> F or W, never Y -> A. A hotspot whose wild type is not a key
# here stops Part 1 rather than falling through to the full alphabet.
CONSERVATIVE = {
    "Y": ("F", "W"),             # aromatic ring kept, hydroxyl traded
    "W": ("F", "Y"),             # the only two that keep the ring
    "F": ("Y", "W"),             # ring kept, hydroxyl or indole added
    "H": ("Y", "W", "N"),        # aromatic, or the amide that keeps the H-bond
    "R": ("K", "Q", "H"),        # charge kept (K, H) or the H-bond without it (Q)
    "N": ("Q", "H"),             # amide lengthened, or aromatic H-bond
    "T": ("S", "V"),             # hydroxyl kept (S) or the beta-branch kept (V)
}

# Parts 3 and 5, the scorers. One `score_<model>` column per model, joined on
# variant_id and never averaged. SCORER drives every ranking; the runners-up are
# extra columns read beside it.
SCORER = "foldx"
RUNNERS_UP = ["antifold", "antifold_seq"]
# A matrix scorer answers a single mutation exactly and has no answer for several,
# so `antifold` is not in the set that scores the combinations. `antifold_seq`
# conditions on the whole mutant and foldx builds it, so both of those can.
SET_SCORERS = [SCORER, "antifold_seq"]
# AntiFold is scored through this repository's `antifold_score`, which needs
# AntiFold's source folder - the one holding models/model.pt. None takes $ANTIFOLD_SRC.
ANTIFOLD_SRC = None
# False is the honest default: every ranking here is SCORER's column. True lets the
# notebook go on with whichever runners-up resolved when SCORER's model is absent.
ALLOW_MISSING_SCORER = False

# Part 4, the gate. Failures are dropped, never weighted against binding score.
GATE_CHECKS = ["liabilities"]

# Part 5, the combinations.
POOL = 25                        # top survivors drawn from
PER_POSITION = 1                 # substitutions per position in the pool: the cap that
                                 # stops one site filling the shortlist
SET_SIZES = (2, 3, 4, 5)         # see the additivity note before Part 5
PER_SIZE = 400                   # generous: the filtering happens after
MIN_DISTANCE = 8.0               # angstroms, CA to CA
SET_SEED = 0
RESCORE_TOP = 50                 # sets carried from the additive first pass into a real score
# True splits those across the set sizes, because an additive sum grows with the number of
# substitutions and a global top N is therefore all of the largest size - which is also why
# `pick_designs` ranks within a size. False takes the global top N instead.
RESCORE_PER_SIZE = True

## 0. Setup and preflight

`preflight` prints one row per adapter this run needs and what was probed for it. It
imports nothing for real and downloads nothing.

Read the `foldx` row carefully. FoldX is licence-gated, and `REGISTRY["foldx"]` is a
`ContainerScorer` with `tool=None`: the package knows the scorer exists and normalizes its
sign and its `ddg` column, and it has no verified command line for it. So
`score_variants(models=["foldx"])` raises `MissingRuntimeError` until `REGISTRY["foldx"]`
is pointed at an adapter that runs your own build - the route column says exactly that.
`examples/foldx_alanine_scan.py` is the working FoldX invocation in this repository, but it
is a script for the alanine scan rather than a registered adapter, so Part 3 cannot borrow
it without writing package code. **Wiring that adapter is a prerequisite for this notebook,
not something it can do for you.**

Each scorer is run in its own `score_variants` call, which is what the docstring
recommends anyway - `**kwargs` and `extra_args` go to every model named, so a flag one
model rejects is an error for the whole call. One call per model also means a scorer that
is absent costs its own column and not the run.

In [ ]:
import json
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

try:  # Jupyter supplies it; this keeps the notebook runnable headless.
    display
except NameError:
    def display(obj):
        print(obj)


def repo_root() -> Path:
    here = Path.cwd().resolve()
    for folder in (here, *here.parents):
        if (folder / "pyproject.toml").is_file():
            return folder
    raise RuntimeError(f"no pyproject.toml above {here}: run this from the repository")


REPO = repo_root()
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from design_pipeline import (  # noqa: E402
    VariantTable,
    enumerate_variants,
    gate,
    get_structure,
    hotspots,
    metrics,
    preflight,
    propose_sets,
    score_variants,
    structures,
)
from design_pipeline.structures import AMINO_ACIDS  # noqa: E402
from design_pipeline.variants import make_id, parse_id, read_table  # noqa: E402

diagnostics = metrics.diagnostics

RUN = Path(RUN)
TABLES = RUN / "tables"
TABLES.mkdir(parents=True, exist_ok=True)
POINTERS = RUN / "pointers.json"

PATHS = {
    "scan": TABLES / "01_scan_with_distance.csv",
    "tiers": TABLES / "01_tiers.csv",
    "requests": TABLES / "02_requests.csv",
    "singles_report": TABLES / "03_single_scores.csv",
    "set_ranking": TABLES / "05_additive_ranking.csv",
    "candidates": TABLES / "06_candidates.csv",
    "manifest": RUN / "manifest.json",
}


def stage(label: str, n: int, **extra) -> None:
    """One `[rows]` line per stage, so a filter that emptied the table is visible here."""
    tail = "  ".join(f"{k}={v}" for k, v in extra.items())
    print(f"[rows] {label:<42} {n:>7}  {tail}".rstrip())


def remember(key: str, table: VariantTable) -> None:
    """Record a variant table's parquet so a later part can reload it without this one."""
    store = json.loads(POINTERS.read_text()) if POINTERS.is_file() else {}
    store[key] = {
        "path": str(table.path),
        "output_dir": str(table.output_dir),
        "structure": str(table.structure) if table.structure else None,
        "rows": table.rows,
        "candidates": table.candidates,
        "columns": list(table.columns),
        "config": json.loads(json.dumps(dict(table.config), default=str)),
    }
    POINTERS.write_text(json.dumps(store, indent=2, sort_keys=True))


def recall(key: str) -> VariantTable:
    """Rebuild the handle from the parquet on disk. The file is the state, not this kernel."""
    if not POINTERS.is_file():
        raise FileNotFoundError(f"{POINTERS} does not exist: run the earlier parts first")
    store = json.loads(POINTERS.read_text())
    if key not in store:
        raise KeyError(
            f"no {key!r} in {POINTERS}; it holds {', '.join(sorted(store)) or 'nothing'}. "
            f"Run the part that writes it."
        )
    record = store[key]
    frame = read_table(Path(record["path"]))
    return VariantTable(
        path=Path(record["path"]),
        output_dir=Path(record["output_dir"]),
        rows=len(frame),
        candidates=int(frame["variant_id"].nunique()),
        columns=tuple(frame.columns),
        structure=Path(record["structure"]) if record["structure"] else None,
        config=record["config"],
    )


print(f"repository {REPO}")
print(f"run        {RUN}\n")

needed = ["pdb_download", SCORER, *RUNNERS_UP, *SET_SCORERS, *GATE_CHECKS]
seen = list(dict.fromkeys(needed))
rows_ = preflight.rows(seen)
print(preflight.render(rows_))
for note in preflight.notes(seen):
    print(f"note: {note}")

RESOLVED = {row[0]: str(row[3]).startswith("yes") for row in rows_}
unresolved = sorted(name for name, ok in RESOLVED.items() if not ok)
print()
if unresolved:
    print(f"unresolved: {', '.join(unresolved)}")
if not RESOLVED.get("pdb_download"):
    raise RuntimeError("pdb_download does not resolve, so there is no parent structure to read")
for check in GATE_CHECKS:
    if not RESOLVED.get(check):
        raise RuntimeError(f"gate check {check!r} does not resolve; Part 4 cannot run")
if not RESOLVED.get(SCORER):
    message = (
        f"SCORER={SCORER!r} does not resolve here, and it is the column every ranking in this "
        f"notebook is taken on. For foldx this means REGISTRY['foldx'].tool is None - the "
        f"package has no verified command line for it - so point REGISTRY['foldx'] at an "
        f"adapter that runs your own build. INSTALL.md has the route."
    )
    if not ALLOW_MISSING_SCORER:
        raise RuntimeError(message)
    warnings.warn(message + " ALLOW_MISSING_SCORER is set, so the run goes on.", RuntimeWarning)

RANK_BY = f"score_{SCORER}"
SCORE_MODELS = list(dict.fromkeys([SCORER, *RUNNERS_UP]))
print(f"\nranking on {RANK_BY}")
print(f"single-mutation columns: {', '.join('score_' + m for m in SCORE_MODELS)}")

### The parent complex

`source=` rather than `pdb_id=`, so this reads the file already in the repository instead
of going to the PDB. `chains=` is checked against the file, which is the line that
prevents the expensive mistake: a wrong chain letter scores every variant at positions
belonging to another molecule, and nothing downstream would notice.

In [ ]:
complex_ = get_structure(
    RUN / "parent",
    source=STRUCTURE_SOURCE,
    chains=(HEAVY, LIGHT, ANTIGEN),
)
STRUCTURE = complex_.path
print(f"structure {STRUCTURE}")

SEQUENCES = {c: structures.chain_sequence(STRUCTURE, c) for c in structures.chain_ids(STRUCTURE)}
ROLES = {HEAVY: "heavy (VH+CH1)", LIGHT: "light (VL+CL)", ANTIGEN: "HER2 ectodomain"}
for chain, seq in SEQUENCES.items():
    print(f"  chain {chain}: {len(seq):>4} residues   {ROLES.get(chain, '?')}")
print(
    "\n1N8Z is a Fab, so chains B and A run past the variable domains into CH1 and CL.\n"
    "The alanine scan covers VH 1-120 and VL 1-107 only, and every position selected below\n"
    "comes from the scan, so no constant-domain position can enter the run."
)

## Part 1 - position selection from the alanine scan

The scan is `trastuzumab_foldx_alanine_scan.csv`: every position of both variable domains
set to alanine, FoldX BuildModel plus AnalyseComplex, three runs per position. `ddg_bind`
is positive when the alanine *weakens* binding, so a large positive value marks a side
chain that carries binding energy.

Two columns of numbering, and only one of them is the one variant ids use:

* `resseq` is the 1N8Z residue number, and for chains A and B of this file it is also the
  1-based chain position - the residues run 1..220 and 1..214 with no insertion codes.
  That is what `enumerate_variants` indexes, so it is what the ids below are built on.
* `scfv_position` is the position in the 242-residue scFv (VH, 15-residue linker, VL). It
  is the key for anything spoken in scFv coordinates and is **not** used to build ids here.

The next cell checks `resseq` and `wt` against the structure for all 227 rows rather than
trusting the claim.

In [ ]:
scan = pd.read_csv(SCAN_CSV)
stage("alanine scan rows", len(scan))

print("\ncoverage")
print(f"  rows                     {len(scan)}")
print(f"  distinct (chain, resseq) {scan.drop_duplicates(['chain', 'resseq']).shape[0]}")
display(
    scan.groupby(["domain", "chain"]).agg(
        positions=("resseq", "size"), first=("resseq", "min"), last=("resseq", "max")
    )
)
print("status")
display(scan["status"].value_counts().to_frame("positions"))
print(
    "`wt_is_ala` is a position that is already alanine: there is no alanine mutant to build,\n"
    "so ddg_bind is empty. Those positions are carried but can never enter a ddG-cut tier."
)

# "Best available": the 10-run doc value where the scan has one, the 3-run mean otherwise.
scan["ddg_bind_best"] = scan["ddg_bind_doc"].where(scan["ddg_bind_doc"].notna(), scan["ddg_bind"])
scan["ddg_runs_best"] = scan["runs_doc"].where(scan["ddg_bind_doc"].notna(), scan["n_runs"])

scored = scan[scan["ddg_bind"].notna()]
stage("positions with a ddg_bind", len(scored))
print("\nddg_bind distribution, kcal/mol (positive = alanine weakens binding)")
display(
    pd.concat(
        {"ddg_bind (3 runs)": scored["ddg_bind"].describe(),
         "ddg_bind_best": scored["ddg_bind_best"].describe()},
        axis=1,
    ).round(3)
)
bands = pd.cut(
    scored["ddg_bind"],
    [-np.inf, -0.2, -1e-9, 1e-9, 0.2, 0.5, 1.0, np.inf],
    labels=["< -0.2", "-0.2..0", "exactly 0", "0..0.2", "0.2..0.5", "0.5..1.0", ">= 1.0"],
)
display(bands.value_counts().reindex(bands.cat.categories).to_frame("positions"))
print(
    "The mass at exactly 0 is the scan's own floor and not a measurement: FoldX reports no\n"
    "interaction-energy change for a position whose side chain is not at the interface."
)

# Where the two ddG columns disagree, which is the one decision TIER_ON makes.
moved = scan[
    scan["ddg_bind_doc"].notna()
    & ((scan["ddg_bind"] >= TIER2_DDG) != (scan["ddg_bind_best"] >= TIER2_DDG))
]
print(f"\npositions that cross the {TIER2_DDG} kcal/mol line between the two columns: {len(moved)}")
if len(moved):
    display(
        moved.assign(id=moved["chain"] + ":" + moved["wt"] + moved["resseq"].astype(str))[
            ["id", "domain", "ddg_bind", "n_runs", "ddg_bind_doc", "runs_doc"]
        ].set_index("id").round(3)
    )
    print(
        "This is why TIER_ON defaults to ddg_bind_best. On the raw 3-run column the hotspot\n"
        "set has nine members and the position above sits inside Tier 1's band - that is, it\n"
        "would be opened to the full alphabet. The 10-run re-run calls it load-bearing."
    )

### The numbering check, and the distance column the scan does not have

The scan has no distance column, so Part 1 computes one: the **minimum heavy-atom distance
from each scanned residue to any atom of chain `ANTIGEN`**. Heavy atom, not CA - a
CA-to-CA cut would call a residue non-interface whose side chain is in the epitope.

`design_pipeline.metrics.interface_residues` answers the boolean form of this question at a
cutoff and returns a set of `(chain, position)`, not a per-residue distance, and nothing in
the package returns the distance itself. The function below is notebook code built on the
package's own `structures.load` and `structures.residue_starts`, so it segments residues
exactly as `chain_sequence` and `ca_coordinates` do - insertion codes included - and its
output is aligned with the chain positions variant ids use.

One inconsistency worth surfacing rather than hiding: the scan was run with HER2 trimmed to
residues 480-607, because FoldX time grows with residue count, while chain C of the
deposited file models 1-607. The saved column is the distance to all of chain C, as asked.
`min_dist_antigen_scanned` beside it is the distance to the 480-607 window the ddG numbers
actually came from, so a position whose two values disagree is visible.

In [ ]:
import biotite.structure as struc

SCAN_WINDOW = (480, 607)   # the HER2 residues the FoldX scan was run against


def min_heavy_atom_distance(structure, chain, partners, residue_range=None):
    """Per position of `chain`, the minimum heavy-atom distance to any atom of `partners`.

    Returns an array aligned with `structures.chain_sequence(structure, chain)`, so index
    i - 1 is chain position i. NaN for a residue with no heavy atom, which a distance
    filter must read as "cannot tell" rather than as zero.
    """
    atoms = structures.load(Path(structure))
    atoms = atoms[struc.filter_amino_acids(atoms)]
    if "element" in atoms.get_annotation_categories():
        atoms = atoms[atoms.element != "H"]
    partner = atoms[np.isin(atoms.chain_id, list(partners))]
    if residue_range is not None:
        low, high = residue_range
        partner = partner[(partner.res_id >= low) & (partner.res_id <= high)]
    if partner.array_length() == 0:
        raise KeyError(f"no atoms for partners {partners} in {structure}")
    sub, starts = structures.residue_starts(atoms, chain)
    bounds = list(starts) + [sub.array_length()]
    out = np.full(len(starts), np.nan)
    target = partner.coord
    for index, (begin, end) in enumerate(zip(bounds, bounds[1:])):
        coords = sub[begin:end].coord
        if not len(coords):
            continue
        out[index] = float(
            np.sqrt(((coords[:, None, :] - target[None, :, :]) ** 2).sum(-1)).min()
        )
    return out


# 1. The numbering claim, checked rather than trusted.
problems = []
for chain in sorted(scan["chain"].unique()):
    sequence = SEQUENCES[chain]
    labels = structures.chain_residue_ids(STRUCTURE, chain)
    for row in scan[scan["chain"] == chain].itertuples():
        position = int(row.resseq)
        if not 0 < position <= len(sequence):
            problems.append(f"{chain}:{row.wt}{position} is outside chain {chain}")
            continue
        if labels[position - 1] != str(position):
            problems.append(
                f"{chain}:{position} has PDB label {labels[position - 1]!r} at chain "
                f"position {position}: resseq is not the chain index in this file"
            )
        if sequence[position - 1] != row.wt:
            problems.append(
                f"{chain}:{row.wt}{position} but the structure has "
                f"{sequence[position - 1]} at chain position {position}"
            )
if problems:
    for line in problems[:20]:
        print(f"  {line}")
    raise RuntimeError(
        f"{len(problems)} numbering mismatches between the scan and {STRUCTURE}. Every id "
        f"built below would point at the wrong residue."
    )
print(f"numbering checked: all {len(scan)} scan rows agree with {STRUCTURE.name} on resseq and wt")

# 2. The distance column.
distances, distances_window = {}, {}
for chain in sorted(scan["chain"].unique()):
    distances[chain] = min_heavy_atom_distance(STRUCTURE, chain, [ANTIGEN])
    distances_window[chain] = min_heavy_atom_distance(
        STRUCTURE, chain, [ANTIGEN], residue_range=SCAN_WINDOW
    )
scan["min_dist_antigen"] = [distances[c][int(p) - 1] for c, p in zip(scan["chain"], scan["resseq"])]
scan["min_dist_antigen_scanned"] = [
    distances_window[c][int(p) - 1] for c, p in zip(scan["chain"], scan["resseq"])
]
scan["id"] = scan["chain"] + ":" + scan["wt"] + scan["resseq"].astype(str)

print(f"\nminimum heavy-atom distance to chain {ANTIGEN}, angstroms")
display(scan[["min_dist_antigen", "min_dist_antigen_scanned"]].describe().round(2))
for cut in (4.0, 5.0, 6.0, 8.0, 10.0):
    stage(f"positions within {cut:>4.1f} A of antigen", int((scan["min_dist_antigen"] < cut).sum()))

gap = scan[(scan["min_dist_antigen_scanned"] - scan["min_dist_antigen"]).abs() > 0.5]
print(
    f"\npositions whose nearest antigen atom lies outside the {SCAN_WINDOW[0]}-{SCAN_WINDOW[1]} "
    f"window the ddG came from: {len(gap)}"
)
# Only the ones close enough for the discrepancy to change a filter are worth printing;
# a framework position 20 A away is unaffected either way.
close_gap = gap[gap["min_dist_antigen"] < 10.0]
if len(close_gap):
    display(
        close_gap[["id", "domain", "ddg_bind", "min_dist_antigen", "min_dist_antigen_scanned"]]
        .set_index("id").round(2)
    )
    print(
        "These are near enough that the window matters: their ddG was computed against a HER2\n"
        "that did not include their nearest atom in the deposited structure."
    )
else:
    print(
        f"  none of them within 10 A, so the discrepancy is confined to distant framework\n"
        f"  positions and cannot move the {TIER1_MAX_DISTANCE} A cut either way."
    )

# A check against the independent record: hotspots.py documents all ten hotspots as within
# 3.8 A of HER2, so the distance column computed here has to agree with that.
recorded = {h.crystal_id: h for h in hotspots.HOTSPOTS if h.status == "hotspot"}
near = scan[scan["id"].isin(recorded)]
print(f"\nthe {len(recorded)} recorded hotspots, by the distance computed here")
display(near[["id", "domain", "ddg_bind", "ddg_bind_best", "min_dist_antigen"]].set_index("id").round(3))
if len(near) != len(recorded):
    raise RuntimeError(
        f"only {len(near)} of the {len(recorded)} hotspots in hotspots.py were found in the "
        f"scan by id; the two are not describing the same numbering"
    )
worst = float(near["min_dist_antigen"].max())
print(f"farthest recorded hotspot: {worst:.2f} A (hotspots.py records all ten within 3.8 A)")
if worst > 4.5:
    warnings.warn(
        f"a recorded hotspot is {worst:.2f} A from chain {ANTIGEN} by this calculation, so the "
        f"chain mapping or the structure is not the one hotspots.py describes",
        RuntimeWarning,
    )

scan.to_csv(PATHS["scan"], index=False)
stage("saved 01_scan_with_distance.csv", len(scan))
print(PATHS["scan"])

### The three tiers

Reloaded from `01_scan_with_distance.csv`, so this cell can be re-run against edited
parameters without recomputing the distances.

**Tier 1, primary targets.** Within `TIER1_MAX_DISTANCE` of the antigen and
`TIER1_DDG[0] <= ddG <= TIER1_DDG[1]`. At the interface and contributing something, but not
load-bearing: a substitution here can gain without risking a contact we already have. Full
alphabet, 19 substitutions per position.

**Tier 2, conservative only.** ddG at or above `TIER2_DDG` - the hotspots. Not excluded,
but restricted to the substitutions `CONSERVATIVE` lists for that wild type, which keep the
feature the scan says carries the binding energy. `enumerate_variants` takes one
`mutations=` alphabet for all of `positions=`, so a per-position alphabet cannot be
expressed that way; Part 2 uses `only=` with explicit ids instead, which is the API's own
route for a specific candidate set and validates every wild type against the structure.

**Tier 3, literature positions.** `TIER3` regardless of alanine ddG, plus `CONTROLS`. Full
alphabet.

A position in more than one tier keeps all three flags, and `tier` resolves the label by
precedence 2 > 1 > 3, so a hotspot stays restricted even if the literature names it.

In [ ]:
scan = pd.read_csv(PATHS["scan"])
stage("reloaded scan", len(scan))

ddg = scan[TIER_ON]
print(f"tiering on {TIER_ON!r}\n")

control_positions = {f"{c}:{w}{p}" for c, p, w, _ in (parse_id(v)[0] for v in CONTROLS)}
scan["is_tier1"] = (
    (scan["min_dist_antigen"] < TIER1_MAX_DISTANCE) & ddg.between(*TIER1_DDG, inclusive="both")
).fillna(False)
scan["is_tier2"] = (ddg >= TIER2_DDG).fillna(False)
scan["is_tier3"] = scan["id"].isin(TIER3) | scan["id"].isin(control_positions)
scan["tier"] = np.select([scan["is_tier2"], scan["is_tier1"], scan["is_tier3"]], [2, 1, 3], default=0)

missing = sorted((set(TIER3) | control_positions) - set(scan.loc[scan["is_tier3"], "id"]))
if missing:
    raise KeyError(
        f"these named positions are not in the scan, or are named with the wrong wild type "
        f"or numbering: {missing}"
    )

SHOW = ["id", "domain", "wt", "ddg_bind", "ddg_bind_best", "ddg_runs_best",
        "min_dist_antigen", "status"]


def show(flag, title):
    rows = scan[scan[flag]].sort_values(["chain", "resseq"])
    stage(title, len(rows))
    display(rows[SHOW].set_index("id").round(3))
    return rows


tier1 = show("is_tier1", f"Tier 1: < {TIER1_MAX_DISTANCE} A, ddG in {TIER1_DDG}")
tier2 = show("is_tier2", f"Tier 2: ddG >= {TIER2_DDG} (hotspots)")
tier3 = show("is_tier3", "Tier 3: literature positions and controls")

# Tier 2 against the independent record, so a disagreement is reported and not a surprise.
recorded = hotspots.positions("hotspot")
found = set(tier2["scfv_position"])
print(f"\nTier 2 holds {len(found)} positions; hotspots.py records {len(recorded)}")
if found != recorded:
    print(f"  only here:        {sorted(found - recorded)} (scFv positions)")
    print(f"  only hotspots.py: {sorted(recorded - found)} (scFv positions)")
    warnings.warn(
        "Tier 2 and design_pipeline.hotspots disagree. hotspots.py holds the 10-run re-runs "
        "from docs/track_a_filtering.md; TIER_ON decides which column is cut here.",
        RuntimeWarning,
    )
else:
    print("  they agree.")

print("\nwhat the scan says about each literature position and control")
display(
    scan[scan["is_tier3"]].assign(
        note=lambda f: np.where(
            f["is_tier2"], "also a hotspot: Tier 2 restriction applies",
            np.where(f["is_tier1"], "also inside Tier 1's band",
                     "outside Tier 1's band and not a hotspot"),
        )
    )[["id", "domain", "ddg_bind", "ddg_bind_best", "min_dist_antigen", "note"]]
    .set_index("id").round(3)
)
print(
    "A negative alanine ddG means FoldX predicts the alanine mutant binds better than the\n"
    "parent, which is a hypothesis and not a measurement. These positions are in the run\n"
    "because the literature names them, which is a different and better reason."
)

# What hotspots.py flags that no tier picked up.
other = [h for h in hotspots.HOTSPOTS if h.status != "hotspot"]
print(f"\nhotspots.py also records {len(other)} non-hotspot positions; where they land here")
display(
    pd.DataFrame(
        [
            {
                "id": h.crystal_id,
                "status": h.status,
                "ddg_bind_doc": h.ddg_bind,
                "tier": (int(scan.loc[scan["id"] == h.crystal_id, "tier"].iloc[0])
                         if (scan["id"] == h.crystal_id).any() else None),
            }
            for h in other
        ]
    ).set_index("id")
)

# A near_miss is a position just under the hotspot line - hotspots.py says check before
# changing it - and Tier 1's band reaches up to TIER2_DDG, so near misses land in Tier 1 and
# are opened to the full alphabet including alanine. That is the band doing what it was
# asked to do, and it is worth seeing how much of Tier 1 it is.
near_miss = {h.crystal_id for h in hotspots.HOTSPOTS if h.status == "near_miss"}
caught = sorted(set(tier1["id"]) & near_miss)
if caught:
    print(
        f"\nnote: {len(caught)} of Tier 1's {len(tier1)} positions are `near_miss` in "
        f"hotspots.py - {', '.join(caught)}.\nThey are under {TIER2_DDG} kcal/mol, so Tier 1's "
        f"band takes them and they get the full alphabet, alanine included.\nLower TIER1_DDG[1] "
        f"(0.9 excludes both) or add them to TIER3 to restrict them instead."
    )

selected = scan[scan["tier"] > 0].sort_values(["tier", "chain", "resseq"])
stage("positions selected overall", len(selected))
display(
    selected["tier"].value_counts().sort_index().rename(
        index={1: "1 primary", 2: "2 hotspot", 3: "3 literature"}
    ).to_frame("positions")
)
selected.to_csv(PATHS["tiers"], index=False)
print(PATHS["tiers"])

### Why the hotspot set and the improvable set barely overlap

Alanine scanning asks **which side chains carry binding energy**. It answers by deleting a
side chain past its first carbon and measuring what binding is lost, so a large positive
ddG means *this contact is doing work*.

The question Part 2 onwards asks is a different one: **which positions improve on
substitution**. A position can be load-bearing and have no room to improve - the side chain
already makes the contact it is going to make, and any substitution is a step down from an
optimum trastuzumab's own maturation already found. A position can also contribute nothing
to an alanine scan and still be the best place to gain, because an alanine scan cannot see
an interaction that is *not yet there*: the gain from putting a tyrosine where a serine sits
is invisible to an experiment that only removes side chains.

So the two sets are expected to be nearly disjoint, and the overlap below is a property of
the method rather than a problem with the data. The practical consequence is the one the
tiers encode: the hotspots are a **constraint** - do not break what works - and Tier 1 is
the **search space**.

In [ ]:
scan = pd.read_csv(PATHS["scan"])

hotspot_set = set(scan.loc[scan[TIER_ON] >= TIER2_DDG, "id"].dropna())
improvable = set(
    scan.loc[
        (scan["min_dist_antigen"] < TIER1_MAX_DISTANCE)
        & scan[TIER_ON].between(*TIER1_DDG, inclusive="both"),
        "id",
    ].dropna()
)
overlap = hotspot_set & improvable

stage("hotspots (load-bearing)", len(hotspot_set))
print("  " + ", ".join(sorted(hotspot_set)))
stage("improvable (Tier 1 search space)", len(improvable))
print("  " + ", ".join(sorted(improvable)))
stage("overlap", len(overlap))
print("  " + (", ".join(sorted(overlap)) if overlap
              else "(none - the two bands are disjoint by construction)"))
print(
    f"\nThe two cuts cannot overlap as written: Tier 1's band tops out at {TIER1_DDG[1]} "
    f"kcal/mol and Tier 2 starts at {TIER2_DDG}.\nThe number that matters is the ratio: "
    f"{len(improvable)} positions to search against {len(hotspot_set)} to protect, out of "
    f"{int(scan['ddg_bind'].notna().sum())} scanned."
)

near = scan[scan["min_dist_antigen"] < TIER1_MAX_DISTANCE]
stage(f"all positions within {TIER1_MAX_DISTANCE} A of antigen", len(near))
display(
    pd.cut(
        near[TIER_ON],
        [-np.inf, 0.0, TIER1_DDG[0], TIER1_DDG[1], np.inf],
        labels=["alanine improves (< 0)", f"0..{TIER1_DDG[0]} (no signal)",
                "Tier 1 band", f">= {TIER2_DDG} (hotspot)"],
    ).value_counts().sort_index().to_frame("positions")
)
print(
    "Read down that column: most interface positions contribute nothing an alanine scan can\n"
    "see. That asymmetry is what this part exists to make explicit before any scoring runs."
)

# The two knobs that decide how big Tier 1 is, so a tuning round can see the cost of each
# before re-running anything downstream.
print("\nwhat widening Tier 1 would buy (positions, before any scoring)")
display(
    pd.DataFrame(
        [
            {
                "max_distance": cut,
                "ddg_low": low,
                "positions": int(
                    (
                        (scan["min_dist_antigen"] < cut)
                        & scan[TIER_ON].between(low, TIER1_DDG[1], inclusive="both")
                    ).sum()
                ),
            }
            for cut in (5.0, 6.0, 8.0)
            for low in (0.2, 0.05, 0.0)
        ]
    ).pivot(index="max_distance", columns="ddg_low", values="positions")
)
print(
    f"The lower bound costs the most. {int(scan[TIER_ON].between(0.0, TIER1_DDG[0], inclusive='left').sum())} "
    f"scanned positions sit in [0, {TIER1_DDG[0]}), and {int((scan[TIER_ON] == 0).sum())} of those\n"
    f"are exactly 0 - the scan's floor rather than a measured zero. Dropping ddg_low to 0.0\n"
    f"therefore stops selecting on contribution at all and selects on distance alone, which is\n"
    f"a different and defensible filter, but not the one Tier 1 describes."
)

## Part 2 - enumerate

Reloaded from `01_tiers.csv`.

`enumerate_variants` has two ways in and only one of them fits. `positions=` plus
`mutations=` applies **one** alphabet to every position named, and the tiers need a
different alphabet per position: the full 19 for Tiers 1 and 3, and only what
`CONSERVATIVE` allows at a Tier 2 hotspot. `only=` takes exactly the candidate ids wanted,
accepts both chains in one call, and checks every id's wild type against the structure, so a
numbering or residue disagreement is an error here rather than a row that silently scores
the wrong thing. That is the route used.

One consequence worth stating: `only=` names the table `requested.parquet` and records
`config["positions"] = None`, because the positions were never passed as positions. The
tier membership that produced the ids is in `01_tiers.csv` and `02_requests.csv` beside it,
which is where to look rather than at the table's own config.

In [ ]:
selected = pd.read_csv(PATHS["tiers"])
stage("reloaded selected positions", len(selected))

ALPHABET = tuple(AMINO_ACIDS)
assert len(ALPHABET) == 20, ALPHABET

requests = []
for row in selected.itertuples():
    if row.tier == 2:
        allowed = CONSERVATIVE.get(row.wt)
        if allowed is None:
            raise KeyError(
                f"{row.id} is a Tier 2 hotspot with wild type {row.wt!r} and CONSERVATIVE has "
                f"no entry for it. Add one - falling through to the full alphabet would put "
                f"{row.wt} -> A on a load-bearing side chain."
            )
    else:
        allowed = ALPHABET
    for mut in allowed:
        if mut == row.wt or row.wt == "X":
            continue
        requests.append(
            {
                "variant_id": make_id([(row.chain, int(row.resseq), row.wt, mut)]),
                "position_id": row.id,
                "chain": row.chain,
                "position": int(row.resseq),
                "wt": row.wt,
                "mut": mut,
                "tier": int(row.tier),
                "domain": row.domain,
                "ddg_bind": row.ddg_bind,
                "ddg_bind_best": row.ddg_bind_best,
                "min_dist_antigen": row.min_dist_antigen,
                "is_control": make_id([(row.chain, int(row.resseq), row.wt, mut)]) in set(CONTROLS),
            }
        )

requests = pd.DataFrame(requests).drop_duplicates("variant_id").reset_index(drop=True)
stage("substitutions requested", len(requests))
display(
    requests.groupby("tier").agg(
        positions=("position_id", "nunique"),
        substitutions=("variant_id", "size"),
        per_position=("variant_id", lambda s: round(len(s) / max(s.nunique(), 1), 1)),
    ).rename(index={1: "1 primary", 2: "2 hotspot", 3: "3 literature"})
)
print("\nTier 2, the restricted lists")
display(
    requests[requests["tier"] == 2]
    .groupby(["position_id", "wt"])["mut"].apply(lambda s: ", ".join(sorted(s)))
    .to_frame("allowed")
)

missing_controls = sorted(set(CONTROLS) - set(requests["variant_id"]))
if missing_controls:
    raise RuntimeError(f"the controls {missing_controls} are not in the requested set")
print(f"\ncontrols present: {', '.join(sorted(set(CONTROLS)))}")

requests.to_csv(PATHS["requests"], index=False)

singles = enumerate_variants(
    complex_,
    chain=HEAVY,                     # the `only=` route resolves whichever chains the ids name
    output_dir=TABLES / "02_singles",
    only=list(requests["variant_id"]),
)
stage("02 singles table", singles.rows, candidates=singles.candidates, path=singles.path.name)
if singles.candidates != len(requests):
    raise RuntimeError(
        f"asked for {len(requests)} candidates and the table holds {singles.candidates}"
    )
remember("singles", singles)
display(singles.frame.head(10))
print(f"\n{singles.path}")

## Part 3 - score the singles

Reloaded from `pointers.json`.

One `score_variants` call per model, chained: each call returns a new table carrying the
columns of the one before it plus its own, so the result accumulates `score_<model>` and
`score_<model>_raw` without anything being averaged. A model that is not configured raises
`MissingRuntimeError` from its own call and costs that column alone - `SCORER`'s absence
has already stopped the run at preflight unless `ALLOW_MISSING_SCORER` was set.

`foldx` is a per-variant scorer, so `score_variants` caps it: `PER_VARIANT_LIMIT` is 2,000
candidates and an uncapped call above that raises rather than queueing a week of compute.
The table here is a few hundred, well under, so no `max_variants` is passed and every
candidate is scored. `antifold` is a matrix scorer - one forward pass over the complex
answers every single mutation - so it costs almost nothing and is in for free.

Sign conventions are normalized at the adapter boundary, so **higher is better binding in
every `score_` column** whatever the tool's own convention; `foldx` carries `sign=-1.0` and
`value_column="ddg"`, and its raw ddG is kept in `score_foldx_raw`.

In [ ]:
singles = recall("singles")
stage("reloaded singles", singles.rows, candidates=singles.candidates)

scored_table = singles
failures = {}
for model in SCORE_MODELS:
    kwargs = {}
    if model.startswith("antifold") and ANTIFOLD_SRC:
        kwargs["antifold_src"] = ANTIFOLD_SRC
    try:
        scored_table = score_variants(
            complex_,
            scored_table,
            output_dir=TABLES / "03_scores" / model,
            models=[model],
            **kwargs,
        )
        column = f"score_{model}"
        got = int(scored_table.frame[column].notna().sum())
        stage(f"scored with {model}", scored_table.candidates, non_null=got)
    except Exception as error:   # one model's absence costs its column, not the run
        failures[model] = f"{type(error).__name__}: {error}"
        print(f"[skip] {model}: {type(error).__name__}: {error}")

if failures:
    print("\nmodels that did not run:")
    for model, why in failures.items():
        print(f"  {model}: {why}")
if RANK_BY not in scored_table.columns:
    raise RuntimeError(
        f"{RANK_BY} is not in the scored table, so there is nothing to rank on. "
        f"{failures.get(SCORER, 'the column is simply absent')}"
    )

remember("scored_singles", scored_table)
stage("03 scored singles", scored_table.rows, candidates=scored_table.candidates)
print(f"columns: {', '.join(scored_table.score_columns)}")
print(scored_table.path)

### Sanity checks on the columns

`beneficial_fraction` is the share of candidates a column calls improving. Most mutations
to an already-matured antibody are neutral or worse, so `PLAUSIBLE_BENEFICIAL` (0.25) is the
loose ceiling: a column reporting 40% beneficial is mis-signed or miscalibrated rather than
optimistic. Note that this table is *not* a random scan - it is a deliberately enriched set
of interface positions - so a somewhat higher fraction here is expected and the check is
against a flipped sign, not against optimism.

`wt_argmax_rate` **cannot be computed from this table**, and the reason is in its own
docstring: it needs the scorer run over the full alphabet *including the wild type*, and
`enumerate_variants` skips wt-to-wt because that is not a variant. Run on the table below it
reports `scored=0`, `skipped=<every position>` and a NaN rate, which is the function
behaving correctly and not a result. It is run anyway, with the counts printed, so the gap
is on the record rather than quietly absent. Getting the real number means a matrix run -
the scorer over every residue at every position - which is `design_pipeline.matrices` and
`notebooks/04_antifold_matrix_check.ipynb`, not this notebook; nothing in the public API
produces a wt row from `enumerate_variants`.

In [ ]:
scored_table = recall("scored_singles")
frame = scored_table.frame
columns = list(scored_table.score_columns)
stage("reloaded scored singles", len(frame), candidates=frame["variant_id"].nunique())
print(diagnostics.CAVEAT + "\n")

shares = diagnostics.beneficial_fraction(frame, columns)
display(
    pd.DataFrame(
        {
            "beneficial_fraction": shares,
            "scored": {diagnostics.scorer_name(c): int(frame[c].notna().sum()) for c in columns},
            "missing": {diagnostics.scorer_name(c): int(frame[c].isna().sum()) for c in columns},
        }
    ).round(3)
)
print(f"PLAUSIBLE_BENEFICIAL = {diagnostics.PLAUSIBLE_BENEFICIAL}")
for name, share in shares.items():
    if share == share and share > diagnostics.PLAUSIBLE_BENEFICIAL:
        print(f"  {name}: {share:.2f} is above the loose ceiling - check the sign convention")

print("\nwt_argmax_rate, for the record (see the markdown above)")
display(
    pd.DataFrame([diagnostics.wt_argmax_rate(frame, c) for c in columns]).set_index("scorer")
)
print(
    f"EXPECTED_WT_ARGMAX = {diagnostics.EXPECTED_WT_ARGMAX}. `scored=0` with every position in\n"
    "`skipped` is the expected output here: this table has no wild-type rows to be the argmax."
)

print("\nhow much the columns agree (Spearman; REDUNDANT_SPEARMAN = "
      f"{diagnostics.REDUNDANT_SPEARMAN})")
pairwise = diagnostics.inter_model_spearman(frame, columns)
if pairwise:
    display(pd.Series({f"{a} vs {b}": v for (a, b), v in pairwise.items()}).round(3).to_frame("rho"))
else:
    print("  only one column ran; nothing to compare")

### Where the controls land

`A:S50F` and `A:S50W` are measured losses. They are in the run so that a column which ranks
them highly is caught here, by a fact we already know, instead of being caught by an assay.
A control in the top decile is not a reason to drop the column on its own - it is two data
points - but it is a reason not to trust that column's ordering of the rest.

In [ ]:
scored_table = recall("scored_singles")
frame = scored_table.frame.drop_duplicates("variant_id").copy()
columns = list(scored_table.score_columns)

for column in columns:
    frame[f"rank_{diagnostics.scorer_name(column)}"] = frame[column].rank(
        ascending=False, method="min"
    )
    frame[f"pct_{diagnostics.scorer_name(column)}"] = frame[column].rank(pct=True)

report = frame.set_index("variant_id")
rank_cols = [c for c in report.columns if c.startswith(("rank_", "pct_"))]
total = int(frame["variant_id"].nunique())

print(f"the controls, out of {total} candidates (rank 1 = best, pct 1.0 = best)")
display(report.loc[[v for v in CONTROLS if v in report.index], columns + rank_cols].round(3))
for column in columns:
    name = diagnostics.scorer_name(column)
    worst = report.loc[[v for v in CONTROLS if v in report.index], f"pct_{name}"].max()
    if worst == worst:
        verdict = "TOP DECILE - this column likes a known loss" if worst >= 0.9 else "fine"
        print(f"  {name}: best control sits at percentile {worst:.2f}  {verdict}")

print(f"\ntop 15 by {RANK_BY}")
display(
    report.sort_values(RANK_BY, ascending=False)
    .head(15)[columns].round(3)
)
report.to_csv(PATHS["singles_report"])
print(f"\n{PATHS['singles_report']}")

## Part 4 - gate

Reloaded from `pointers.json`.

`gate` drops what fails, and that is the whole point: a failure is never traded off against
binding score, because a tenfold affinity gain is not worth a new glycosylation sequon and
the only way to be sure the trade is not made is to make it impossible. The annotated
table - every candidate, every `pass_<check>` and `reason_<check>`, failures included - is
written beside the survivors, so a gate that dropped too much can be re-tuned from disk
without re-running anything.

`liabilities` is pure sequence regex, so it runs on everything and needs no cap.

The second cell asks whether the gate preferentially killed the high scorers.
`gate_score_correlation` is the rank correlation between a check's verdict and a score
column: negative means the gate is removing the candidates the scorer liked most, which is a
real tension between binding and developability and is worth surfacing rather than silently
absorbing - the shortlist that comes out is then the second-best-binding set, and nobody
said so.

In [ ]:
scored_table = recall("scored_singles")
stage("reloaded scored singles", scored_table.rows, candidates=scored_table.candidates)

survivors = gate(
    scored_table,
    checks=GATE_CHECKS,
    output_dir=TABLES / "04_gated",
    complex_=complex_,
)
annotated_path = Path(survivors.config["annotated"])
annotated = read_table(annotated_path)

stage("into the gate", scored_table.candidates)
stage("04 survivors", survivors.candidates)
stage("dropped", scored_table.candidates - survivors.candidates)
remember("survivors", survivors)

print("\npass rate per check")
display(pd.Series(diagnostics.pass_rate(annotated, GATE_CHECKS)).round(3).to_frame("pass_rate"))

for check in GATE_CHECKS:
    candidates = annotated.drop_duplicates("variant_id")
    failed = candidates[~candidates[f"pass_{check}"].astype(bool)]
    print(f"\n{check}: {len(failed)} failures, by reason")
    if len(failed):
        display(failed[f"reason_{check}"].value_counts().to_frame("candidates"))
        display(failed.groupby("mut").size().sort_values(ascending=False).to_frame("failures").head(10))

print(f"\n{survivors.path}")
print(annotated_path)

In [ ]:
survivors = recall("survivors")
annotated = read_table(Path(json.loads(POINTERS.read_text())["survivors"]["config"]["annotated"]))
columns = list(survivors.score_columns)

print("does the gate preferentially kill high scorers?")
print("(Spearman between the pass verdict and each score column; negative = yes)\n")
correlation = diagnostics.gate_score_correlation(annotated, columns, GATE_CHECKS)
display(
    pd.Series({f"{check} vs {scorer}": v for (check, scorer), v in correlation.items()})
    .round(3).to_frame("rho")
)
for (check, scorer), rho in correlation.items():
    if rho != rho:
        print(f"  {check} vs {scorer}: NaN - the check passed or failed everything, so there is "
              f"no correlation to take")
    elif rho < -0.1:
        print(f"  {check} vs {scorer}: {rho:.2f} - the gate is removing what {scorer} liked most")

candidates = annotated.drop_duplicates("variant_id")
for column in columns:
    passed = candidates[candidates[f"pass_{GATE_CHECKS[0]}"].astype(bool)][column]
    failed = candidates[~candidates[f"pass_{GATE_CHECKS[0]}"].astype(bool)][column]
    print(
        f"\n{column}: passed mean {passed.mean():.3f} (n={passed.notna().sum()}), "
        f"failed mean {failed.mean():.3f} (n={failed.notna().sum()})"
    )

print("\nsurvivors by tier (joined back to the requested set)")
requests = pd.read_csv(PATHS["requests"])
joined = survivors.frame.drop_duplicates("variant_id").merge(
    requests[["variant_id", "tier", "position_id", "domain"]], on="variant_id", how="left"
)
display(
    joined.groupby("tier").agg(survivors=("variant_id", "size"), positions=("position_id", "nunique"))
    .rename(index={1: "1 primary", 2: "2 hotspot", 3: "3 literature"})
)

## Part 5 - combinations

Reloaded from `pointers.json`.

`propose_sets` draws random greedy sets from the best survivors. Three parameters carry the
decisions:

* **`pool=POOL`, `per_position=PER_POSITION`.** The shortlist is the best `POOL` survivors
  after keeping at most `PER_POSITION` substitution per position, which is the cap that
  stops one site filling the shortlist. On a real table this matters a lot: the docstring
  records that on 1N8Z the best 30 singles by the table-route AntiFold score sat on **8**
  positions, one of them contributing ten substitutions - a pool from which no set of 9 can
  be drawn at all. `per_position=1` is also the package default.
* **`min_distance=MIN_DISTANCE`, 8.0 A CA to CA.** Consecutive CAs are about 3.8 A apart, so
  a 4.0 A cut excludes adjacent residues and constrains nothing else. **One correction to
  the premise: 8.0 is already this version's default** - `propose_sets(min_distance: float =
  8.0)` - and the 4.0 the docstring argues against was the earlier default. It is passed
  explicitly here anyway, because a parameter this consequential should be visible in the
  call and not inherited.
* **`sizes=SET_SIZES`, `per_size=PER_SIZE`.** Generous, and filtered after.
  `config["found"]` reports how many of each size were actually drawn, which is the number to
  read: a size the conflict graph cannot support comes back short rather than failing, and the
  interface positions here are clustered, so the larger sizes are the ones that will fall
  short at 8 A.

`propose_sets` drops every `score_` column on the way out, deliberately: a set's score is not
a function of its singles' scores. The ranking in the next cell puts one back by hand, under
a justification that is specific to this system and written out there.

In [ ]:
survivors = recall("survivors")
stage("reloaded survivors", survivors.rows, candidates=survivors.candidates)
if RANK_BY not in survivors.columns:
    raise RuntimeError(f"{RANK_BY} is not in the survivors table; Part 3 did not produce it")

sets = propose_sets(
    survivors,
    sizes=SET_SIZES,
    per_size=PER_SIZE,
    output_dir=TABLES / "05_sets",
    by=RANK_BY,
    pool=POOL,
    per_position=PER_POSITION,
    min_distance=MIN_DISTANCE,
    seed=SET_SEED,
)
remember("sets", sets)
config = dict(sets.config)
stage("05 proposed sets", sets.rows, candidates=sets.candidates)
print(
    f"\npool {config['pool']} singles on {config['positions_in_pool']} positions, "
    f"min_distance {config['min_distance']} A, seed {config['seed']}"
)
display(
    pd.DataFrame(
        {
            "asked": {int(k): PER_SIZE for k in SET_SIZES},
            "found": {int(k): int(v) for k, v in config["found"].items()},
        }
    ).rename_axis("size")
)
short = {int(k): int(v) for k, v in config["found"].items() if int(v) < PER_SIZE}
if short:
    n_pool = int(config["pool"])
    incompatible = sum(int(v) for v in config["conflicts"].values()) // 2
    pairs = n_pool * (n_pool - 1) // 2 - incompatible
    print(
        f"sizes that came back short: {short}\n"
        f"A size that cannot be filled is reported, not raised, and there are two separate "
        f"reasons for it.\n"
        f"  1. The pool admits fewer DISTINCT sets of that size than per_size={PER_SIZE}. With "
        f"{n_pool} singles and {incompatible}\n     incompatible pairs there are only {pairs} valid "
        f"pairs in existence, which is the hard ceiling on size 2.\n"
        f"  2. The greedy draw could not reach the size inside its attempt budget, which is what "
        f"happens at the\n     larger sizes once the {config['min_distance']} A rule leaves too few "
        f"mutually distant substitutions.\n"
        f"Both are visible in `found` above; neither is an error."
    )

print("\nthe pool, and how constrained each member is")
print("(conflicts = how many other pool members this one can never share a set with)")
conflicts = pd.Series(config["conflicts"], dtype=float).astype(int).sort_values(ascending=False)
display(conflicts.to_frame("conflicts").head(15))

print("\nhow often each substitution appears across the proposed sets")
usage = sets.frame.groupby(["chain", "position", "wt", "mut"]).size().sort_values(ascending=False)
display(usage.to_frame("in_sets").head(15))
print(f"\n{sets.path}")

### The additive first pass, and why it is allowed here

**Ranking a set by the sum of its singles' scores is normally forbidden in this package.**
`propose_sets` drops the score columns for exactly this reason, `score_candidates` returns
no score for a multi-mutation candidate rather than adding two marginals, and
`mutant_sequence`'s docstring says the only honest way to score a set is one pass over the
full mutant. Nothing below changes that: the sum computed here is a **cheap first pass whose
only job is to choose which `RESCORE_TOP` sets get scored properly**, and no additive number
reaches the output as a score.

The justification for using it as a *sorting key* is specific to this system and does not
generalise. On the ADAPT panel, epistasis across the measured combinations has an SD of
**0.186 kcal/mol** against a median measurement SD of **0.165 kcal/mol**, and **0 of 11**
combinations sit more than 2 SD from zero epistasis. Interaction between substitutions is
therefore at or below the noise floor of the measurements it would have to be detected in, so
additivity is a defensible estimator on this system for substitutions that are spatially
separated - which `min_distance=8.0` A is there to enforce.

Three caveats that travel with that:

1. **Eleven combinations is a small panel.** "0 of 11 beyond 2 SD" is consistent with real
   epistasis the panel is too small to resolve. The claim is that epistasis is small, not
   that it is zero.
2. **It holds for spatially separated substitutions.** Two changes in contact would
   rearrange each other, and a sum cannot see that. The 8 A rule is what keeps the sets
   inside the regime the panel measured.
3. **Error accumulates with set size.** A sum of five singles carries five singles' worth of
   error, so the additive ranking is a weaker instrument at size 5 than at size 2. That is
   one reason `SET_SIZES` stops at 5 rather than the package default of 5 to 10.

The cell after this one re-scores the top sets as whole mutants, and Part 6 reports how well
the additive ordering predicted the real one - the check this assumption deserves on this
run, not just on the panel.

In [ ]:
sets = recall("sets")
survivors = recall("survivors")
stage("reloaded sets", sets.rows, candidates=sets.candidates)

KEY = ["chain", "position", "wt", "mut"]
single_scores = survivors.frame.drop_duplicates("variant_id")[KEY + list(survivors.score_columns)]
members = sets.frame.merge(single_scores, on=KEY, how="left", validate="many_to_one")

missing = members[members[RANK_BY].isna()]
if len(missing):
    raise RuntimeError(
        f"{len(missing)} set members have no {RANK_BY}; propose_sets drew a single the "
        f"survivors table cannot score, which should be impossible"
    )

ranking = members.groupby("variant_id").agg(
    n_mutations=("n_mutations", "first"),
    additive=(RANK_BY, "sum"),
    worst_member=(RANK_BY, "min"),
    best_member=(RANK_BY, "max"),
)
for column in survivors.score_columns:
    if column != RANK_BY:
        ranking[f"additive_{diagnostics.scorer_name(column)}"] = (
            members.groupby("variant_id")[column].sum(min_count=1)
        )
ranking["additive_mean"] = ranking["additive"] / ranking["n_mutations"]
ranking = ranking.sort_values("additive", ascending=False)
ranking["rank_overall"] = ranking["additive"].rank(ascending=False, method="min").astype(int)
ranking["rank_in_size"] = (
    ranking.groupby("n_mutations")["additive"].rank(ascending=False, method="min").astype(int)
)
stage("sets ranked additively", len(ranking))

print(f"\nadditive sum of {RANK_BY}, by set size")
display(ranking.groupby("n_mutations")["additive"].describe().round(3))
print("\ntop 15 sets by the additive first pass, overall")
display(ranking.head(15).round(3))

# The bias this ranking has, stated before it is used: an additive sum grows with the number
# of substitutions whenever the singles are mostly positive, so a global top N is the largest
# size and nothing else. The counts below say how badly.
overall_top = ranking.head(RESCORE_TOP)
print(f"\nwhat a GLOBAL top {RESCORE_TOP} would hold, by size")
display(overall_top["n_mutations"].value_counts().sort_index().to_frame("sets"))
if overall_top["n_mutations"].nunique() == 1:
    print(
        f"  One size, as expected. That is why RESCORE_PER_SIZE exists: `pick_designs` ranks\n"
        f"  within a size for the same reason - scores are not comparable across sizes, because\n"
        f"  every extra substitution moves the sum."
    )

if RESCORE_PER_SIZE:
    sizes_present = sorted(ranking["n_mutations"].unique())
    quota = max(1, RESCORE_TOP // len(sizes_present))
    chosen = ranking[ranking["rank_in_size"] <= quota]
    print(f"\nRESCORE_PER_SIZE: {quota} per size over {len(sizes_present)} sizes")
else:
    chosen = overall_top
    print(f"\nRESCORE_PER_SIZE is off: the global top {RESCORE_TOP}")

ranking.to_csv(PATHS["set_ranking"])
top_ids = list(chosen.sort_values("additive", ascending=False).index)
stage("carried into the re-score", len(top_ids))
display(chosen["n_mutations"].value_counts().sort_index().to_frame("sets carried"))
print(f"\n{PATHS['set_ranking']}")

### Re-score the top sets as whole mutants

`enumerate_variants(only=...)` rebuilds a variant table from the set ids - that is the one
place the function accepts multi-mutation ids, and it checks every wild type against the
structure on the way. Then `score_variants` scores each set as a single candidate.

`SET_SCORERS` leaves `antifold` out on purpose. A matrix scorer is exact for one mutation and
has **no answer for several**: `score_candidates` returns `None` for any candidate with more
than one mutation rather than summing marginals, so an `antifold` column here would be
entirely empty. `antifold_seq` asks AntiFold's own `score_sequence_in_complex` how likely the
whole mutant chain is given the backbone, and `foldx` builds the mutant, so both answer a set.

In [ ]:
ranking = pd.read_csv(PATHS["set_ranking"], index_col="variant_id")
if RESCORE_PER_SIZE:
    quota = max(1, RESCORE_TOP // ranking["n_mutations"].nunique())
    chosen = ranking[ranking["rank_in_size"] <= quota]
else:
    chosen = ranking.nsmallest(RESCORE_TOP, "rank_overall")
top_ids = list(chosen.sort_values("additive", ascending=False).index)
stage("reloaded additive ranking", len(ranking), carried=len(top_ids))
display(chosen["n_mutations"].value_counts().sort_index().to_frame("sets carried"))

top_sets = enumerate_variants(
    complex_,
    chain=HEAVY,
    output_dir=TABLES / "05_top_sets",
    only=top_ids,
)
stage("top sets as a variant table", top_sets.rows, candidates=top_sets.candidates)

rescored = top_sets
set_failures = {}
for model in SET_SCORERS:
    kwargs = {}
    if model.startswith("antifold") and ANTIFOLD_SRC:
        kwargs["antifold_src"] = ANTIFOLD_SRC
    try:
        rescored = score_variants(
            complex_,
            rescored,
            output_dir=TABLES / "05_rescored" / model,
            models=[model],
            **kwargs,
        )
        column = f"score_{model}"
        # Per candidate, not per row: a set is several rows sharing one variant_id.
        per_candidate = rescored.frame.drop_duplicates("variant_id")[column]
        stage(f"re-scored with {model}", rescored.candidates,
              scored=int(per_candidate.notna().sum()))
    except Exception as error:
        set_failures[model] = f"{type(error).__name__}: {error}"
        print(f"[skip] {model}: {type(error).__name__}: {error}")

if set_failures:
    print("\nmodels that did not score the sets:")
    for model, why in set_failures.items():
        print(f"  {model}: {why}")

remember("rescored_sets", rescored)
stage("05 re-scored sets", rescored.rows, candidates=rescored.candidates)
print(f"columns: {', '.join(rescored.score_columns)}")
print(rescored.path)

## Part 6 - the output, and what the additive pass was worth

Reloaded from `pointers.json` and `05_additive_ranking.csv`.

The last check is the one this run can actually make about its own shortcut: the rank
correlation between the additive first pass and the real whole-mutant score, on the sets
that got both. It is reported **within each set size** as well as overall, because the two
answer different questions - across sizes the additive sum is partly just a count of
substitutions, so a correlation taken over the pooled sets is inflated by set length rather
than by additivity working.

A high within-size correlation means the cheap pass chose well, for *these* sets, which were
already selected by that pass - so it is an upper bound on how well additivity orders the
sets it rejected, not an estimate. A low one means the top of the additive list was not the
top of the real list, and `RESCORE_TOP` should be larger next round.

The output table is ranked on the **re-scored** column, never on the additive sum. The
additive columns are kept beside it so the two orderings can be compared, labelled
`additive_*` so nothing downstream can mistake one for a score.

In [ ]:
rescored = recall("rescored_sets")
ranking = pd.read_csv(PATHS["set_ranking"], index_col="variant_id")

frame = rescored.frame
set_columns = [c for c in rescored.score_columns if frame[c].notna().any()]
if not set_columns:
    raise RuntimeError("no scorer produced a column for the sets; there is nothing to rank")
SET_RANK_BY = f"score_{SCORER}" if f"score_{SCORER}" in set_columns else set_columns[0]
print(f"ranking the sets on {SET_RANK_BY}")

per_set = frame.drop_duplicates("variant_id").set_index("variant_id")[set_columns]
per_set["n_mutations"] = frame.groupby("variant_id").size()
per_set = per_set.join(ranking[[c for c in ranking.columns if c.startswith("additive")]])
stage("sets with a real score", int(per_set[SET_RANK_BY].notna().sum()), of=len(per_set))

usable = per_set[per_set[SET_RANK_BY].notna() & per_set["additive"].notna()]
rho = metrics.spearman(usable["additive"].to_numpy(float), usable[SET_RANK_BY].to_numpy(float))
print(f"\nadditive first pass vs {SET_RANK_BY}, over {len(usable)} sets: Spearman {rho:.3f}")
print("within each size, where the sum is not confounded by set length:")
display(
    pd.DataFrame(
        [
            {
                "n_mutations": int(size),
                "sets": len(group),
                "spearman": metrics.spearman(
                    group["additive"].to_numpy(float), group[SET_RANK_BY].to_numpy(float)
                ),
            }
            for size, group in usable.groupby("n_mutations")
        ]
    ).set_index("n_mutations").round(3)
)
print(
    "These are the sets the additive pass already chose, so this is an upper bound on how well\n"
    "additivity orders the sets it rejected. A low number here means RESCORE_TOP is too small."
)

candidates = per_set.sort_values(SET_RANK_BY, ascending=False)
candidates.insert(0, "rank", range(1, len(candidates) + 1))
# And the rank within the size, which is the one `pick_designs` would use: every extra
# substitution moves a whole-mutant score, so the overall rank is partly a size ranking.
candidates.insert(
    1,
    "rank_in_size",
    candidates.groupby("n_mutations")[SET_RANK_BY]
    .rank(ascending=False, method="min")
    .astype("Int64"),
)
candidates.insert(2, "mutations", [
    " ".join(f"{c}:{w}{p}{m}" for c, p, w, m in parse_id(str(v))) for v in candidates.index
])
print(f"\nbest set of each size by {SET_RANK_BY}")
display(candidates[candidates["rank_in_size"] == 1].round(3))
print(f"\ntop 20 candidates by {SET_RANK_BY}, overall")
display(candidates.head(20).round(3))
candidates.to_csv(PATHS["candidates"])

print("\nposition diversity and chain balance of the full set list")
display(pd.Series(diagnostics.position_diversity(frame)).to_frame("value"))
display(pd.Series(diagnostics.chain_balance(frame)).round(3).to_frame("share"))
print(f"\n{PATHS['candidates']}")

In [ ]:
store = json.loads(POINTERS.read_text())
manifest = {
    "run": str(RUN),
    "structure": str(STRUCTURE),
    "chains": {"heavy": HEAVY, "light": LIGHT, "antigen": ANTIGEN},
    "scorer": SCORER,
    "runners_up": RUNNERS_UP,
    "set_scorers": SET_SCORERS,
    "rank_by": RANK_BY,
    "set_rank_by": SET_RANK_BY,
    "tier_on": TIER_ON,
    "tier_parameters": {
        "tier1_max_distance": TIER1_MAX_DISTANCE,
        "tier1_ddg": list(TIER1_DDG),
        "tier2_ddg": TIER2_DDG,
        "tier3": list(TIER3),
        "controls": list(CONTROLS),
        "conservative": {k: list(v) for k, v in CONSERVATIVE.items()},
    },
    "set_parameters": {
        "pool": POOL,
        "per_position": PER_POSITION,
        "sizes": list(SET_SIZES),
        "per_size": PER_SIZE,
        "min_distance": MIN_DISTANCE,
        "seed": SET_SEED,
        "rescore_top": RESCORE_TOP,
        "rescore_per_size": RESCORE_PER_SIZE,
    },
    "scorers_unavailable": {**failures, **{f"sets:{k}": v for k, v in set_failures.items()}},
    "tables": {key: PATHS[key].as_posix() for key in PATHS if PATHS[key].is_file()},
    "variant_tables": {
        key: {k: record[k] for k in ("path", "rows", "candidates")} for key, record in store.items()
    },
    "additive_vs_rescored_spearman": float(rho),
}
PATHS["manifest"].write_text(json.dumps(manifest, indent=2, default=str))

print("row counts, end to end")
for key in ("singles", "scored_singles", "survivors", "sets", "rescored_sets"):
    if key in store:
        stage(key, store[key]["rows"], candidates=store[key]["candidates"])
print()
print(json.dumps(manifest["scorers_unavailable"], indent=2) if manifest["scorers_unavailable"]
      else "every scorer named ran")
print(f"\n{PATHS['manifest']}")

### What this notebook does not do

* **It stops at a ranked set list.** `pick_designs` is the next call - it enforces the rules
  that only hold across designs taken together (each substitution in two to four designs,
  designs differing at three positions or more, a quota per size, an isoelectric-point
  bound), and `frontier` reports the best score at each mutation count. Neither is run here
  because both need the number of designs to be submitted, which is not a filtering decision.
* **No structure veto.** `notebooks/03_end_to_end.ipynb` re-folds the designs and drops the
  ones whose predicted interface falls below the parent's seed noise. That is the check which
  catches a set that scores well and does not fold, and it is a separate run.
* **No accuracy claim.** Every number printed here is either a prediction or a diagnostic
  computed from this run's own output. Nothing in it has been compared against a measurement,
  so nothing in it says the predictions are good - only the ground-truth metrics in the
  harness do that.